<h1>2D reflection and transmission against photon frequency</h1>

In [ ]:
#General imports
resol = 300
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 1.0,
    "lines.markersize": 4,
})

import sys
import numpy as np
from pathlib import Path
pi = np.pi

cwd = Path().resolve()
project_root = next(path for path in (cwd, *cwd.parents) if (path / "src").exists())
sys.path.append(str(project_root))
sys.path.append(str(project_root / "2d_setup" / "scripts_experiments"))

from src.fiber_geometry import load_geometry, plot_geometry
from reflection_vs_frequency import result_filename, run_reflection_vs_frequency


<h2>Load the geometry</h2>

In [ ]:
kind = "slab"  # "slab" or "disk"
lengths = [50.0, 1.0]
matter_width = 1.0
matter_height = 1.0
N_A = 20
N_layer = 1  # ignored for disk geometries

geometry = load_geometry(kind, lengths, matter_width, matter_height, N_A, N_layer)
atom_positions = geometry["atom_positions"]
L1, L2 = geometry["lengths"]

fig, ax = plt.subplots(figsize=(8.0, 2.2), dpi=resol)
plot_geometry(kind, lengths, matter_width, matter_height, N_A, N_layer, ax=ax)
ax.set_title(f"Loaded geometry: {geometry['label']}")
plt.show()

print(f"Loaded geometry from: {geometry['path']}")


<h2>Execute the experiment</h2>

Each experiment corresponds to a given frequency window $[\omega_{\min}, \omega_{\max}]$.

In [ ]:
#Parameters of the simulation
Omega = 10*np.pi
d = 1.0
Omega_A = Omega * np.ones(geometry['N_A'])
g_A = d * np.ones(geometry['N_A'], dtype=float)
param_atom = {
    'Omega_A': Omega_A,
    'g_A': g_A,
    'lengths': [L1, L2],
    'positions': atom_positions,
}
param_photon_bis = {'Deltap0': [0.15*np.pi, 0.15*np.pi], 'x0': [-geometry['L1'] / 4, 0.0]}
param_time_evol = {'T': 50, 'dt': 0.05}

#Define an array of frequencies around the matter resonance
nb_freq = 21
omega_init_tab = np.linspace(Omega - 2*abs(d)**2,
                            Omega + 2*abs(d)**2,
                            nb_freq)

#Assess reflection/transmission for different frequency windows
frequency_windows = [
    (0*pi, 20*pi),
    (0*pi, 14*pi),
    (8*pi, 14*pi),
    (9*pi, 11*pi)
]

Run the experiment for each frequency window.

In [ ]:
for i in range(len(frequency_windows)):
    index_experiment = i + 1
    print("Running frequency window number ", index_experiment, " out of ", len(frequency_windows))
    cutoffs = {'omega_min': frequency_windows[i][0], 'omega_max': frequency_windows[i][1]}

    frequency_values, final_reflection_tab, final_transmission_tab = run_reflection_vs_frequency(
        param_photon_bis,
        param_atom,
        param_time_evol,
        omega_init_tab,
        cutoffs,
        geometry,
        index_experiment=index_experiment,
        progress=True,
    )

<h2>Plot the results</h2>

In [ ]:
data_dir = project_root / "2d_setup" / "results" / "csv_files" / "reflection_vs_frequency"

colors_reflection = ["#081f5c", "#a73c51", "#bda546", "#009E73"]
markers_reflection = ["o", "s", "^", "D"]

fig, ax = plt.subplots(figsize=(3, 3), dpi=resol)

for index_experiment in range(1, len(frequency_windows) + 1):
    omega_min = frequency_windows[index_experiment - 1][0]
    omega_max = frequency_windows[index_experiment - 1][1]
    data_file = data_dir / result_filename(geometry, index_experiment)
    df = pd.read_csv(data_file)

    photon_frequency_tab = df["photon_frequency_tab"].to_numpy()
    reflection_tab = df["final_reflection_tab"].to_numpy()
    transmission_tab = df["final_transmission_tab"].to_numpy()
    reflection_plus_transmission = reflection_tab + transmission_tab

    label_window = rf"$[{omega_min/np.pi:.1f}\pi,{omega_max/np.pi:.1f}\pi]$"
    color = colors_reflection[index_experiment - 1]
    marker = markers_reflection[index_experiment - 1]

    print(
        rf"{label_window}: R+T = "
        f"[{reflection_plus_transmission.min():.6f}, {reflection_plus_transmission.max():.6f}] "
        f"(mean = {reflection_plus_transmission.mean():.6f})"
    )

    ax.plot(
        photon_frequency_tab / np.pi,
        reflection_tab,
        marker=marker,
        markersize=3,
        color=color,
        linewidth=1.0,
        alpha=0.95,
        label=rf"$\mathcal{{R}}$ {label_window}",
        zorder=3,
    )

omega_min_pi = np.min(omega_init_tab) / np.pi
omega_max_pi = np.max(omega_init_tab) / np.pi
ax.set_xlim(omega_min_pi - 0.1, omega_max_pi + 0.1)

ax.set_xlabel(r"\textbf{Photon frequency} $\omega_{\rm init}/\pi$", fontsize=10)
ax.set_ylabel(r"\textbf{Reflection probability} $\mathcal{R}$", fontsize=10)
ax.set_ylim(-0.05, 1.05)
ax.grid(color="0.9", linestyle="-", linewidth=0.4)
ax.tick_params(axis="both", which="major", length=4, width=0.8, labelsize=10)
ax.legend(prop={"size": 9}, loc="lower center", frameon=False, ncol=1, handlelength=1.5)
plt.tight_layout()
plt.show()
